In [ ]:
# Import necessary packages
from pydantic import BaseModel, ValidationError, Field, EmailStr
from typing import List, Literal, Optional
import json
from datetime import date
from dotenv import load_dotenv
import instructor
import anthropic

### Define your Pydantic models for user input and LLM output

In [ ]:
# Define the UserInput model for customer support queries
class UserInput(BaseModel):
    name: str
    email: EmailStr
    query: str
    order_id: Optional[int] = Field(
        None,
        description="5-digit order number (cannot start with 0)",
        ge=10000,
        le=99999
    )
    purchase_date: Optional[date] = None

# Define the CustomerQuery model that inherits from UserInput
class CustomerQuery(UserInput):
    priority: str = Field(
        ..., description="Priority level: low, medium, high"
    )
    category: Literal[
        'refund_request', 'information_request', 'other'
    ] = Field(..., description="Query category")
    is_complaint: bool = Field(
        ..., description="Whether this is a complaint"
    )
    tags: List[str] = Field(..., description="Relevant keyword tags")

### Provide sample input and validate it using your model

In [ ]:
# Define your input data as a JSON string
user_input_json = '''{
    "name": "Joe User",
    "email": "joe.user@example.com",
    "query": "I ordered a new computer monitor and it arrived with the screen cracked. This is the second time this has happened. I need a replacement ASAP.",
    "order_id": 12345,
    "purchase_date": "2025-12-31"
}'''

In [ ]:
# Validate the user_input_json by creating a UserInput instance
user_input = UserInput.model_validate_json(user_input_json)

### Build a prompt and call the Anthropic API with the instructor package for structured output

In [ ]:
prompt = (
    f"Analyze the following customer query {user_input} "
    f"and provide a structured response."
)

In [ ]:
# Load environment variables
load_dotenv()
# Use Anthropic with Instructor to get structured output
anthropic_client = instructor.from_anthropic(
    anthropic.Anthropic()
)

response = anthropic_client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=2048,
    messages=[
        {
            "role": "user", 
            "content": prompt
        }
    ],
    response_model=CustomerQuery  
)

In [ ]:
# Inspect the returned structured data
print(type(response))
print(response.model_dump_json(indent=2))

### Use Anthropic's structured output API with your Pydantic schema

In [ ]:
# Initialize Anthropic client and call passing CustomerQuery in your API call
client = anthropic.Anthropic()
response = client.messages.parse(
    model="claude-sonnet-4-6",
    max_tokens=2048,
    messages=[{"role": "user", "content": prompt}],
    output_format=CustomerQuery
)
response_content = response.content[0].text
print(type(response_content))
print(response_content)

### Additional advanced usage and inspection

In [ ]:
# Validate the repsonse you got from the LLM
valid_data = CustomerQuery.model_validate_json(
    response_content
)
print(type(valid_data))
print(valid_data.model_dump_json(indent=2))

In [ ]:
# Get the parsed Pydantic object directly from the Anthropic response
response = client.messages.parse(
    model="claude-sonnet-4-6",
    max_tokens=2048,
    messages=[{"role": "user", "content": prompt}],
    output_format=CustomerQuery
)

print(type(response))
print(type(response.parsed_output))
print(response.parsed_output.model_dump_json(indent=2))

In [ ]:
# Investigate class inheritance structure of the Anthropic response
def print_class_inheritence(llm_response):
    for cls in type(llm_response).mro():
        print(f"{cls.__module__}.{cls.__name__}")

print_class_inheritence(response)

In [ ]:
# Print the response type and content 
print(type(response.parsed_output))
print(response.parsed_output.model_dump_json(indent=2))

In [ ]:
# Try out the Pydantic AI package for defining an agent and getting a structured response
from pydantic_ai import Agent

agent = Agent(
    model="anthropic:claude-sonnet-4-6",
    output_type=CustomerQuery,
)

response = await agent.run(prompt)

In [ ]:
# Print out the repsonse type and content
print(type(response.output))
print(response.output.model_dump_json(indent=2))